Más análisis exploratorio de los datos...

In [1]:
import pandas as pd 
import numpy as np

In [2]:
catalogo_productos = pd.read_csv("Datos-finales/catalogo_productos.csv")
especificaciones_cajas = pd.read_csv("Datos-finales/especificaciones_cajas.csv")
operaciones_planta = pd.read_csv("Datos-finales/operaciones_planta.csv")
procurement_cajas = pd.read_csv("Datos-finales/procurement_cajas.csv")

Agregamos las cargas máximas para cada tipo de caja (pasamos de mm a m)

In [3]:
#Calculo el perímetro. Divido por 1000 para pasar a m
perimetro_cajas = 2 * (especificaciones_cajas["caja_exterior_largo"] + especificaciones_cajas["caja_exterior_ancho"])/ 1000 

ect_por_grosor = {2.5: 600, 2.7: 730, 3.0: 1000, 4.1: 1200, 4.5: 1400, 4.6: 1450, 4.7: 1500, 4.8: 1550, 5.0: 1650}

ects = [ect_por_grosor[g] for g in especificaciones_cajas['caja_grosor_mm']]

especificaciones_cajas['carga_max'] = ects * perimetro_cajas / 9.81

Averigüemos si la carga soportada actualmente se acerca a la máxima

¿cuenta el peso del producto en la base de la pila?

In [4]:
merged = catalogo_productos.merge(
    especificaciones_cajas[['caja_tipo_id', 'cantidad_cajas_alto', 'carga_max', 'caja_grosor_mm',
                            'caja_exterior_largo','caja_exterior_ancho','caja_exterior_alto',
                            'caja_interior_largo','caja_interior_ancho','caja_interior_alto']],
    on='caja_tipo_id',
    how='left'
)
#El -1 es para NO contar la caja de la base
catalogo_productos['carga_soportada'] = merged['peso_neto_caja'] * (merged['cantidad_cajas_alto'] - 1)

In [5]:
catalogo_productos[catalogo_productos['carga_soportada']/ merged['carga_max'] > 1]

,codigo_producto,descripcion_producto,ingrediente_forma,tipo_proyecto,subcategoria,categoria,tamaño_corte,peso_neto_caja,tamaño_paquete,cantidad_paquetes,peso_neto_paquete,caja_tipo_id,carga_soportada


El numero máximo de cajas que se pueden apilar para un producto dado es

$$n_{max\_carga} = \lfloor(carga\_max/peso\_neto\_paquete)\rfloor + 1 $$

mientras este número no haga que la pila sopbrepase los 1800 mm

In [6]:
merged[merged['caja_exterior_alto'] * merged['cantidad_cajas_alto'] >= 1800]

,codigo_producto,descripcion_producto,ingrediente_forma,tipo_proyecto,subcategoria,categoria,tamaño_corte,peso_neto_caja,tamaño_paquete,cantidad_paquetes,...,caja_tipo_id,cantidad_cajas_alto,carga_max,caja_grosor_mm,caja_exterior_largo,caja_exterior_ancho,caja_exterior_alto,caja_interior_largo,caja_interior_ancho,caja_interior_alto


In [7]:
merged['n_por_ancho_pallet'] = (1200 // merged['caja_exterior_ancho']).astype(int)   # 1200mm / ancho caja
merged['n_por_largo_pallet'] = (800 // merged['caja_exterior_largo']).astype(int)   # 800mm / largo caja
merged['n_capas_max_altura'] = (1800 // merged['caja_exterior_alto']).astype(int) # limitado por altura

In [8]:
merged['n_max_carga'] = ((merged['carga_max'] // merged['peso_neto_caja']) + 1).astype(int)

El n_max de cajas que se pueden apilar es el mínimo entre el número máximo que soporta el tipo de caja y el máximo número antes de que sobrepase los 1800 mm de altura

In [9]:
merged['n_max'] = np.minimum(
    merged['n_capas_max_altura'],
    merged['n_max_carga']
)

In [10]:
merged[merged['n_capas_max_altura'] <= merged['n_max_carga']].shape

(435, 26)

Siempre gana la altura! Se puede bajar la carga_max entonces... probemos con el mínimor grosor permitido = 3.0 mm

In [11]:
nuevo_per = 2 * (merged["caja_interior_largo"] + merged["caja_interior_ancho"] + 12.0)/ 1000
merged['max_carga_3mm'] = 1000 * nuevo_per / 9.81
merged['n_max_carga_3mm'] = ((merged['max_carga_3mm'] // merged['peso_neto_caja']) + 1).astype(int)

In [12]:
merged['n_capas_max_altura_3mm'] = (1800 // (merged['caja_interior_alto'] + 6.0)).astype(int)

In [13]:
merged[merged['n_capas_max_altura_3mm'] <= merged['n_max_carga_3mm']].shape

(435, 29)

Sigue ganando altura!

In [14]:
merged[merged['n_capas_max_altura_3mm'] > merged['n_capas_max_altura']].shape

(63, 29)

Además en algunos casos ganamos una capa más

In [15]:
merged['n_por_ancho_pallet_3mm'] = (1200 // (merged['caja_interior_ancho'] + 6.0)).astype(int)
merged['n_por_largo_pallet_3mm'] = (800 // (merged['caja_interior_largo'] + 6.0)).astype(int)

In [16]:
merged[merged['n_por_ancho_pallet_3mm'] == merged['n_por_ancho_pallet']].shape

(435, 31)

In [17]:
merged[merged['n_por_ancho_pallet_3mm'] == merged['n_por_ancho_pallet']].shape

(435, 31)

In [18]:
ids = merged[merged['n_por_ancho_pallet_3mm'] < merged['n_por_largo_pallet']]['caja_tipo_id']
print(ids.shape)
especificaciones_cajas[especificaciones_cajas['caja_tipo_id'].isin(ids)]['caja_grosor_mm'].value_counts()

(0,)


Series([], Name: count, dtype: int64)

En ancho no ganamos nada... en largo tampoco y de hecho "perdimos" aunque en realidad esos 5 son con grosor menor el cual no está permitido

Si asumimos que el volumen interno actual de las cajas ES el volumen del envase primario entonces:

In [19]:
catalogo_productos['producto_largo'] = merged['caja_interior_largo']
catalogo_productos['producto_ancho'] = merged['caja_interior_ancho']
catalogo_productos['producto_alto'] = merged['caja_interior_alto']

Y el headspace del producto se calcula como:

In [20]:
catalogo_productos['headspace_largo'] = merged['caja_interior_largo'] - catalogo_productos['producto_largo'] 
catalogo_productos['headspace_ancho'] = merged['caja_interior_ancho'] - catalogo_productos['producto_ancho'] 
catalogo_productos['headspace_alto'] = merged['caja_interior_alto'] - catalogo_productos['producto_alto'] 

El cual debe cumplir con una restricción dependiendo del grosor de la caja y con un tope absoluto de 40 mm:

In [21]:
max_headspace = {2.5: 0.06, 2.7: 0.06, 3.0: 0.06, 4.1: 0.08, 4.5: 0.08, 4.6: 0.1, 4.7: 0.1, 4.8: 0.1, 5.0: 0.1}

headspace_cols = [c for c in catalogo_productos.columns if c.startswith('headspace')]

for c in headspace_cols:
    dim = c.replace('headspace_', '')
    col_name = f'max_headspace_{dim}'
    
    max_hs_por_caja = [max_headspace[g] for g in merged['caja_grosor_mm']]
    
    dim_int_caja = merged[f'caja_interior_{dim}']
    
    catalogo_productos[col_name] = dim_int_caja * max_hs_por_caja
    
    #El máximo es 40 mm
    catalogo_productos[catalogo_productos[col_name] > 40.0] = 40.0

In [23]:
procurement_cajas

,caja_tipo_id,volumen_tipo_planta_buenos_aires,volumen_tipo_planta_curitiba,volumen_tipo_planta_santiago,volumen_tipo_planta_monterrey,volumen_tipo_planta_bakersfield,costo_unitario_base,costo_total_base,costo_unitario_planta_buenos_aires,costo_unitario_planta_curitiba,...,descuento_planta_curitiba,descuento_planta_santiago,descuento_planta_monterrey,descuento_planta_bakersfield,costo_por_pallet,descuento_planta_buenos_aires_original,descuento_planta_curitiba_original,descuento_planta_santiago_original,descuento_planta_monterrey_original,descuento_planta_bakersfield_original
0,016d196c89dcfcb306b853a776a879b9,0,334398,39939,0,0,0.70,262035.90,0.770,0.560,...,-0.2,0.0,0.1,0.1,150,+10%,-20%,+10%,+10%,+10%
1,01a2a319402ed2155292c04d8748e16f,417,950,621,313,370,0.70,1869.70,0.770,0.770,...,0.1,0.1,0.1,0.1,150,+10%,"+10,00%",+10%,+10%,+10%
2,026560e43f3fc6afe0ce89d7ddf61626,0,0,35599,0,0,0.65,23139.35,0.715,0.715,...,0.1,0.0,0.1,0.1,150,+10%,+10%,+0%,+10%,+10%
3,02d7c6680102bd11e067c00c9b71ab9c,0,58364,0,0,0,0.70,40854.80,0.770,0.630,...,-0.1,0.1,0.1,0.1,150,+10%,-10%,+10 %,+10%,+10%
4,0378f85c226113f4ac40fd360217bb8a,0,360373,0,97776,517960,0.65,634470.85,0.715,0.520,...,-0.2,0.1,-0.1,-0.3,150,+10,20%,+10%,-10%,-30%
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
199,f609bd5b4ffc901bae0837e2befee31d,22584,0,0,0,0,0.70,15808.80,0.700,0.770,...,0.1,0.1,0.1,0.1,150,+0%,+10%,+10%,+10%,10%
200,fc5516122a48681b6a5d3d70768ee02a,0,4121,0,0,0,0.65,2678.65,0.715,0.715,...,0.1,0.1,0.1,0.1,150,+10%,+10%,+10%,+10%,+10%
201,fcc3ee3ebc86c553e5f951274d02868e,0,0,14653,0,0,0.65,9524.45,0.715,0.715,...,0.1,0.1,0.1,0.1,150,+10%,+10%,+10%,+10%,+10%
202,fce36b1b860f566c05d3e06484e6f4e2,122276,64806,35602,0,0,0.70,155878.80,0.560,0.630,...,-0.1,0.0,0.1,0.1,150,-10%,+0%,+10%,+10%,+10%


In [ ]:
op_prod = catalogo_productos.merge(operaciones_planta,
                                   on='codigo_producto')

In [44]:
operaciones_planta.columns

Index(['codigo_producto', 'volumen_producto_canal_servicios_comida',
       'volumen_producto_canal_minorista',
       'volumen_producto_canal_cadenas_corporativas',
       'volumen_producto_canal_otros', 'volumen_producto_total',
       'volumen_producto_planta_buenos_aires',
       'volumen_producto_planta_curitiba', 'volumen_producto_planta_santiago',
       'volumen_producto_planta_monterrey',
       'volumen_producto_planta_bakersfield',
       'cantidad_pallets_planta_buenos_aires',
       'cantidad_pallets_planta_curitiba', 'cantidad_pallets_planta_santiago',
       'cantidad_pallets_planta_monterrey',
       'cantidad_pallets_planta_bakersfield', 'cantidad_pallets_total',
       'costo_total_planta_buenos_aires', 'costo_total_planta_curitiba',
       'costo_total_planta_santiago', 'costo_total_planta_monterrey',
       'costo_total_planta_bakersfield', 'costo_pallets_planta_buenos_aires',
       'costo_pallets_planta_curitiba', 'costo_pallets_planta_santiago',
       'costo_pal

El volumen que aparece en "operaciones_planta" es consistente consigo mismo:

In [ ]:
cols = [c for c in operaciones_planta.columns if c.startswith('volumen_producto_planta_')]
vol_tot_calculado = operaciones_planta[cols].sum(axis=1)

operaciones_planta[~np.isclose(vol_tot_calculado,operaciones_planta['volumen_producto_total'])].shape

(0, 29)

In [52]:
cols = [c for c in operaciones_planta.columns if c.startswith('volumen_producto_canal_')]
vol_tot_calculado = operaciones_planta[cols].sum(axis=1)

operaciones_planta[~np.isclose(vol_tot_calculado,operaciones_planta['volumen_producto_total'])].shape

(0, 29)

In [55]:
procurement_cajas.columns

Index(['caja_tipo_id', 'volumen_tipo_planta_buenos_aires',
       'volumen_tipo_planta_curitiba', 'volumen_tipo_planta_santiago',
       'volumen_tipo_planta_monterrey', 'volumen_tipo_planta_bakersfield',
       'costo_unitario_base', 'costo_total_base',
       'costo_unitario_planta_buenos_aires', 'costo_unitario_planta_curitiba',
       'costo_unitario_planta_santiago', 'costo_unitario_planta_monterrey',
       'costo_unitario_planta_bakersfield', 'descuento_planta_buenos_aires',
       'descuento_planta_curitiba', 'descuento_planta_santiago',
       'descuento_planta_monterrey', 'descuento_planta_bakersfield',
       'costo_por_pallet', 'descuento_planta_buenos_aires_original',
       'descuento_planta_curitiba_original',
       'descuento_planta_santiago_original',
       'descuento_planta_monterrey_original',
       'descuento_planta_bakersfield_original', 'volumen_tipo_total'],
      dtype='str')

Entre "operaciones_planta" y "procurement_cajas" hay inconsistencias:

In [53]:
volumen_por_caja = op_prod.groupby('caja_tipo_id')['volumen_producto_total'].sum().reset_index()
volumen_por_caja = volumen_por_caja.rename(columns={'volumen_producto_total': 'volumen_total'})

In [54]:
cols = [c for c in procurement_cajas.columns if c.startswith('volumen_tipo_planta_')]
procurement_cajas['volumen_tipo_total'] = procurement_cajas[cols].sum(axis=1)

In [74]:
condicion = ~np.isclose(procurement_cajas['volumen_tipo_total'], volumen_por_caja['volumen_total'])
procurement_cajas[condicion]

,caja_tipo_id,volumen_tipo_planta_buenos_aires,volumen_tipo_planta_curitiba,volumen_tipo_planta_santiago,volumen_tipo_planta_monterrey,volumen_tipo_planta_bakersfield,costo_unitario_base,costo_total_base,costo_unitario_planta_buenos_aires,costo_unitario_planta_curitiba,...,descuento_planta_santiago,descuento_planta_monterrey,descuento_planta_bakersfield,costo_por_pallet,descuento_planta_buenos_aires_original,descuento_planta_curitiba_original,descuento_planta_santiago_original,descuento_planta_monterrey_original,descuento_planta_bakersfield_original,volumen_tipo_total
0,016d196c89dcfcb306b853a776a879b9,0,334398,39939,0,0,0.70,262035.90,0.770,0.560,...,0.0,0.1,0.1,150,+10%,-20%,+10%,+10%,+10%,374337
4,0378f85c226113f4ac40fd360217bb8a,0,360373,0,97776,517960,0.65,634470.85,0.715,0.520,...,0.1,-0.1,-0.3,150,+10,20%,+10%,-10%,-30%,976109
6,057b7f8c3772129fb0b8cccf3c827c80,15062,132568,22446,11304,13350,0.65,126574.50,0.715,0.520,...,0.0,0.1,0.1,150,+10%,-10%,+10%,+10%,+10%,194730
7,079828584749f89c6e998317886ea4cf,2630,5986,158904,1974,2330,0.70,120276.80,0.770,0.770,...,-0.2,0.1,0.1,150,10%,+10%,-10%,+10%,+10%,171824
8,093e0a1389e0cc0f644ceea470a3cc01,109265,79365,0,0,0,0.60,113178.00,0.480,0.540,...,0.1,0.1,0.1,150,-20%,-10%,+ 10%,+10%,+10%,188630
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
190,ef0525f0911fd36073c2b10f4d81dfd5,0,21735263,6904843,0,6568595,0.65,22885655.65,0.715,0.455,...,-0.3,0.1,-0.3,150,+10%,30%,-30%,+10%,-30%,35208701
191,ef2aa69f2a021c5eb8f04cae5d3e66f3,113020,191744,95828,132548,0,0.65,346541.00,0.520,0.520,...,-0.1,-0.2,0.1,150,+0%,+0%,+0%,+ 0%,+10%,533140
192,efbf9fb4de26ed4bd82dc7987c339263,0,1877934,0,0,2492424,0.65,2840732.70,0.715,0.455,...,0.1,0.1,-0.3,150,+10%,-20%,+10%,+10%,-20%,4370358
202,fce36b1b860f566c05d3e06484e6f4e2,122276,64806,35602,0,0,0.70,155878.80,0.560,0.630,...,0.0,0.1,0.1,150,-10%,+0%,+10%,+10%,+10%,222684


In [75]:
volumen_por_caja[condicion]

,caja_tipo_id,volumen_total
0,016d196c89dcfcb306b853a776a879b9,124779
4,0378f85c226113f4ac40fd360217bb8a,1971428
6,057b7f8c3772129fb0b8cccf3c827c80,97365
7,079828584749f89c6e998317886ea4cf,85912
8,093e0a1389e0cc0f644ceea470a3cc01,364592
...,...,...
190,ef0525f0911fd36073c2b10f4d81dfd5,6360351
191,ef2aa69f2a021c5eb8f04cae5d3e66f3,133285
192,efbf9fb4de26ed4bd82dc7987c339263,728393
202,fce36b1b860f566c05d3e06484e6f4e2,111342


In [76]:
op_prod

,codigo_producto,descripcion_producto,ingrediente_forma,tipo_proyecto,subcategoria,categoria,tamaño_corte,peso_neto_caja,tamaño_paquete,cantidad_paquetes,...,costo_total_planta_bakersfield,costo_pallets_planta_buenos_aires,costo_pallets_planta_curitiba,costo_pallets_planta_santiago,costo_pallets_planta_monterrey,costo_pallets_planta_bakersfield,costo_pallets_total,costo_total,suma_canales,check_canales_ok
0,BR0001,"5 bolsas de 2,5 kg | Bastones rectos de brocol...",Bastones rectos de brocoli,Estacional,Bastones clasicos - Reserva Privada,Bastones clasicos,Fino,12.50,"5 X 2,5 KG",5,...,0.00,0,5369100,99750,331200,0,5800050,775782.560,1546613,True
1,BR0002,Espirales de brocoli - clasico - Caja de 13 pa...,Espirales de brocoli,Forma estable,Bastones clasicos - Sigilo,Bastones clasicos,Forma,9.75,"13 X 0,75 KG",13,...,0.00,0,435150,0,0,0,435150,72389.720,139211,True
2,BR0003,Gajos de brocoli - sazonado - Caja de 13 packs...,Gajos de brocoli,Forma estable,Bastones clasicos - Sigilo,Bastones clasicos,Forma,9.75,"13 X 0,75 KG",13,...,96603.36,0,0,0,0,323550,323550,96603.360,172506,True
3,BR0004,Bastones rectos de brocoli - crocante - Caja d...,Bastones rectos de brocoli,Estacional,Bastones clasicos - Sigilo,Bastones clasicos,Grueso,11.25,"9 X 1,5 KG",9,...,141291.80,0,0,0,0,727950,727950,141291.800,271715,True
4,BR0005,"13 bolsas de 0,75 kg | Espirales de brocoli si...",Espirales de brocoli,Forma estable,Bastones clasicos - Sigilo,Bastones clasicos,Forma,9.75,"13 X 0,75 KG",13,...,0.00,0,23850,0,0,0,23850,4248.160,7586,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
458,BR0417,Mix crujiente de brocoli - clasico - Caja de 1...,Mix crujiente de brocoli,Forma estable,Bastones clasicos - Sigilo,Bastones clasicos,Forma,8.25,"11 X 0,75 KG",11,...,0.00,0,0,7500,0,0,7500,2125.970,2761,True
459,BR0418,"BonsaiGreen Mix crujiente de brocoli 11 x 0,75 kg",Mix crujiente de brocoli,Forma estable,Bastones clasicos - Sigilo,Bastones clasicos,Forma,8.25,"11 X 0,75 KG",11,...,0.00,0,0,10650,0,0,10650,3035.340,3942,True
460,BR0419,"5 bolsas de 2,5 kg | Bastones rectos de brocol...",Bastones rectos de brocoli,Estacional,Bastones clasicos - Reserva Privada,Bastones clasicos,Corte steak,12.50,"5 X 2,5 KG",5,...,0.00,0,0,0,90300,0,90300,28145.000,43300,True
461,BR0420,"5 bolsas de 2,5 kg | Bastones rectos de brocol...",Bastones rectos de brocoli,Estacional,Bastones clasicos - Reserva Privada,Bastones clasicos,Medio,12.50,"5 X2,5 KG",5,...,0.00,0,56850,0,425700,0,482550,95237.415,205852,True
